# Tests de integración de OP-11 `read_flows`

Notebook orientado a verificar el comportamiento end-to-end de la operación pública
OP-11 `read_flows`, considerando:

- lectura formal desde bundles `.golondrina`;
- reconstrucción de `FlowDataset`;
- coherencia de `summary`, `parameters`, evento y metadata;
- recuperación controlada desde sidecars parcialmente degradados;
- lectura opcional de `flow_to_trips`;
- política post-read de `metadata["is_validated"] = False`;
- soporte de bundles persistidos en Parquet y Feather;
- fallas fatales por layouts o sidecars inconsistentes.

Algunos casos usan previamente `write_flows()` para construir un artefacto formal válido
que luego es leído por `read_flows()`. En esos casos, el foco del test sigue siendo
la reconstrucción y el contrato observable de OP-11.

Este notebook contiene únicamente integration tests de `read_flows`.
No incluye helper-level tests ni smoke tests.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:

- imports generales;
- imports del módulo;
- helpers de testing reutilizables;
- fixtures de `FlowDataset` suficientemente ricas;
- carpeta local para artefactos persistidos;
- y configuración básica de display.

Los artefactos de los tests se escriben bajo una carpeta local
`./tmp_integration_read_flows`, ubicada en la misma raíz del notebook.

### 0.1 Imports generales

Qué prepara: imports base, utilidades de filesystem y deep copy para no contaminar fixtures.

In [1]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd

### 0.2 Imports del módulo

In [2]:
from pylondrina.datasets import FlowDataset
from pylondrina.errors import ExportError
from pylondrina.io.flows import (
    write_flows,
    read_flows,
    WriteFlowsOptions,
    ReadFlowsOptions,
)

### 0.3 Helpers de testing reutilizables

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def issue_codes(report) -> list[str]:
    return [issue.code for issue in report.issues]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def sort_df(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    return df.sort_values(by=by).reset_index(drop=True)


def assert_df_equal_untyped(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    by: list[str],
) -> None:
    pd.testing.assert_frame_equal(
        sort_df(left, by),
        sort_df(right, by),
        check_dtype=False,
        check_categorical=False,
    )


INTEGRATION_ROOT = Path("./tmp_integration_read_flows")
ARTIFACTS_ROOT = INTEGRATION_ROOT / "artifacts"


def reset_integration_root() -> Path:
    if INTEGRATION_ROOT.exists():
        shutil.rmtree(INTEGRATION_ROOT)
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    return INTEGRATION_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = ARTIFACTS_ROOT / case_name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def artifact_aux_filename(storage_format: str) -> str:
    if storage_format == "parquet":
        return "flow_to_trips.parquet"
    if storage_format == "feather":
        return "flow_to_trips.feather"
    raise ValueError(f"storage_format no soportado: {storage_format!r}")

### 0.4 Fixtures de flows ricas para integración

Qué prepara:

- una tabla de flows con varias columnas analíticas y de segmentación;
- un auxiliar `flow_to_trips` opcional;
- `FlowDataset` realista con `aggregation_spec`, `metadata`, `events`,
  `provenance` y `source_trips` vivo en memoria.

In [4]:
ORIGINS = [
    "8828308281fffff",
    "8828308283fffff",
    "8828308285fffff",
    "8828308287fffff",
]

DESTINATIONS = [
    "8828308291fffff",
    "8828308293fffff",
    "8828308295fffff",
    "8828308297fffff",
    "8828308299fffff",
]

MODES = ["bus", "metro", "car"]
PURPOSES = ["work", "education", "shopping", "leisure"]
DAY_TYPES = ["weekday", "weekend"]
GENDERS = ["female", "male"]
INCOME_Q = ["1", "3", "5"]
TIME_PERIODS = ["morning_peak", "midday", "afternoon_peak"]


def make_rich_flows_df(*, repeat_blocks: int = 1) -> pd.DataFrame:
    rows = []
    base_ts = pd.Timestamp("2026-04-01T06:00:00Z")

    idx = 0
    for rep in range(repeat_blocks):
        for origin in ORIGINS:
            for destination in DESTINATIONS:
                for mode in MODES:
                    for day_type in DAY_TYPES:
                        for gender in GENDERS:
                            purpose = PURPOSES[idx % len(PURPOSES)]
                            income_q = INCOME_Q[idx % len(INCOME_Q)]
                            time_period = TIME_PERIODS[idx % len(TIME_PERIODS)]

                            window_start = (
                                base_ts
                                + pd.Timedelta(hours=(idx % 10))
                                + pd.Timedelta(days=rep)
                            )
                            window_end = window_start + pd.Timedelta(hours=1)

                            flow_count = 5 + (idx % 17)
                            flow_value = round(
                                flow_count
                                * (
                                    1.0
                                    + (
                                        0.15
                                        if mode == "metro"
                                        else 0.05
                                        if mode == "bus"
                                        else 0.25
                                    )
                                ),
                                3,
                            )

                            rows.append(
                                {
                                    "flow_id": f"f_{rep:02d}_{idx:05d}",
                                    "origin_h3_index": origin,
                                    "destination_h3_index": destination,
                                    "flow_count": int(flow_count),
                                    "flow_value": float(flow_value),
                                    "mode": mode,
                                    "purpose": purpose,
                                    "day_type": day_type,
                                    "user_gender": gender,
                                    "income_quintile": income_q,
                                    "time_period": time_period,
                                    "window_start_utc": window_start,
                                    "window_end_utc": window_end,
                                    "avg_trip_weight": round(
                                        0.8 + (idx % 9) * 0.21,
                                        3,
                                    ),
                                    "segment_label": f"{mode}|{day_type}|{gender}",
                                }
                            )
                            idx += 1

    return pd.DataFrame(rows)


def make_flow_to_trips_df(
    flows_df: pd.DataFrame,
    *,
    links_per_flow: int = 3,
) -> pd.DataFrame:
    rows = []
    movement_counter = 0

    for _, row in flows_df.iterrows():
        for _ in range(links_per_flow):
            movement_counter += 1
            rows.append(
                {
                    "flow_id": row["flow_id"],
                    "movement_id": f"m_{movement_counter:07d}",
                }
            )

    return pd.DataFrame(rows)


def make_rich_flowdataset(
    *,
    repeat_blocks: int = 1,
    with_trip_links: bool = False,
    validated: bool = False,
    dataset_id: str = "flow-dset-integration-001",
) -> FlowDataset:
    flows_df = make_rich_flows_df(repeat_blocks=repeat_blocks)
    flow_to_trips_df = (
        make_flow_to_trips_df(flows_df)
        if with_trip_links
        else None
    )

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode", "day_type", "user_gender"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "min_trips_per_flow": 1,
    }

    metadata = {
        "dataset_id": dataset_id,
        "is_validated": bool(validated),
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-01T12:00:00Z",
                "parameters": {
                    "h3_resolution": 8,
                    "group_by": ["mode", "day_type", "user_gender"],
                    "time_aggregation": "hour",
                    "time_basis": "origin",
                    "min_trips_per_flow": 1,
                },
                "summary": {
                    "n_flows": int(len(flows_df)),
                    "n_trips_in": int(len(flows_df) * 4),
                    "n_trips_aggregated": int(len(flows_df) * 4),
                    "n_trips_dropped": 0,
                    "n_flow_to_trips_rows": (
                        int(len(flow_to_trips_df))
                        if flow_to_trips_df is not None
                        else None
                    ),
                },
                "issues_summary": {
                    "counts": {
                        "info": 0,
                        "warning": 0,
                        "error": 0,
                    },
                    "top_codes": [],
                },
            }
        ],
        "notes": {"fixture": "integration_rich_flowdataset"},
    }

    provenance = {
        "derived_from": [
            {
                "source_type": "trips",
                "dataset_id": "trip-dset-origin-001",
                "schema_version": "1.1",
            }
        ],
        "prior_events_summary": {"n_events": 3},
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=aggregation_spec,
        source_trips={"debug": "in_memory_only"},
        metadata=metadata,
        provenance=provenance,
    )

### 0.5 Inicialización de entorno y fixtures reutilizables

In [5]:
reset_integration_root()

flowdataset_small = make_rich_flowdataset(
    repeat_blocks=1,
    with_trip_links=False,
    validated=False,
    dataset_id="flow-dset-small-001",
)

flowdataset_with_trip_links = make_rich_flowdataset(
    repeat_blocks=1,
    with_trip_links=True,
    validated=False,
    dataset_id="flow-dset-links-001",
)

print("INTEGRATION_ROOT =", INTEGRATION_ROOT.resolve())
print("ARTIFACTS_ROOT =", ARTIFACTS_ROOT.resolve())
print("flowdataset_small.flows.shape =", flowdataset_small.flows.shape)
print("flowdataset_with_trip_links.flows.shape =", flowdataset_with_trip_links.flows.shape)
print(
    "flowdataset_with_trip_links.flow_to_trips.shape =",
    flowdataset_with_trip_links.flow_to_trips.shape,
)

display(flowdataset_small.flows.head(3))
show_ok("Sección 0 - setup y fixtures ricas para integración de OP-11")

INTEGRATION_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_integration_read_flows
ARTIFACTS_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_integration_read_flows\artifacts
flowdataset_small.flows.shape = (240, 15)
flowdataset_with_trip_links.flows.shape = (240, 15)
flowdataset_with_trip_links.flow_to_trips.shape = (720, 2)


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female


OK - Sección 0 - setup y fixtures ricas para integración de OP-11


### 0.6 Configuración de display

In [6]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Bloque 1 - read feliz desde bundle formal Parquet rico

Qué prueba:

- camino principal correcto de `read_flows`;
- sidecar obligatorio;
- fallback automático a `.golondrina`;
- reconstrucción de:
  - `flows`,
  - `aggregation_spec`,
  - `provenance`,
  - metadata persistida;
- `source_trips=None` post-read;
- `metadata["is_validated"] = False`;
- evento `read_flows`;
- summary y parameters coherentes.

Se usa `write_flows()` solo para construir un bundle Parquet formal que luego
es leído por OP-11.

In [7]:
case_dir = make_case_dir("case_01_read_happy_parquet")
artifact_path = case_dir / "flows_read_happy"

flows = copy.deepcopy(flowdataset_small)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,  # sin sufijo; debe usar fallback a .golondrina
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

effective_root = Path(str(artifact_path) + ".golondrina")

assert read_report.ok is True

# Parameters efectivos
assert read_report.parameters["path"] == str(effective_root)
assert read_report.parameters["strict"] is False
assert read_report.parameters["keep_metadata"] is True
assert read_report.parameters["read_flow_to_trips"] is False

# Summary
assert read_report.summary["n_flows"] == len(flows.flows)
assert read_report.summary["n_columns"] == len(flows.flows.columns)
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert set(read_report.summary["files_read"]) == {
    "flows.parquet",
    "flows.metadata.json",
}

# Dataset reconstruido
assert_df_equal_untyped(
    loaded.flows,
    flows.flows,
    by=["flow_id"],
)

assert loaded.aggregation_spec == flows.aggregation_spec
assert loaded.provenance == flows.provenance
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

# Evento read
event = loaded.metadata["events"][-1]
assert event["op"] == "read_flows"
assert event["parameters"] == read_report.parameters
assert event["summary"] == read_report.summary
assert "issues_summary" in event

display(read_report)
show_ok("Bloque 1 - read feliz desde bundle formal Parquet rico")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_columns': 15, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_0aa9fc9f-1513-467d-967e-f891516f19a0'}, parameters={'path': 'tmp_integration_read_flows\\artifacts\\case_01_read_happy_parquet\\flows_read_happy.golondrina', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': False})

OK - Bloque 1 - read feliz desde bundle formal Parquet rico


## Bloque 2 - read con auxiliar solicitado y existente

Qué prueba:

- lectura correcta de `flow_to_trips` cuando fue persistido;
- `read_flow_to_trips=True`;
- `flow_to_trips_loaded=True`;
- conteo correcto de filas auxiliares;
- igualdad estructural entre el auxiliar original y el reconstruido.

Se usa un bundle Parquet formal para ejercitar una ruta explícita distinta
de los casos Feather posteriores.

In [8]:
case_dir = make_case_dir("case_02_read_with_aux_parquet")
artifact_path = case_dir / "flows_with_aux"

flows = copy.deepcopy(flowdataset_with_trip_links)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert read_report.ok is True
assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)
assert loaded.flow_to_trips is not None

assert_df_equal_untyped(
    loaded.flows,
    flows.flows,
    by=["flow_id"],
)

assert_df_equal_untyped(
    loaded.flow_to_trips,
    flows.flow_to_trips,
    by=["flow_id", "movement_id"],
)

display(loaded.flows)
display(loaded.metadata)
display(read_report.summary)
show_ok("Bloque 2 - read con auxiliar solicitado y existente")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,f_00_00235,8828308287fffff,8828308299fffff,19,21.85,metro,leisure,weekend,male,3,midday,2026-04-01 11:00:00+00:00,2026-04-01 12:00:00+00:00,1.01,metro|weekend|male
236,f_00_00236,8828308287fffff,8828308299fffff,20,25.00,car,work,weekday,female,5,afternoon_peak,2026-04-01 12:00:00+00:00,2026-04-01 13:00:00+00:00,1.22,car|weekday|female
237,f_00_00237,8828308287fffff,8828308299fffff,21,26.25,car,education,weekday,male,1,morning_peak,2026-04-01 13:00:00+00:00,2026-04-01 14:00:00+00:00,1.43,car|weekday|male
238,f_00_00238,8828308287fffff,8828308299fffff,5,6.25,car,shopping,weekend,female,3,midday,2026-04-01 14:00:00+00:00,2026-04-01 15:00:00+00:00,1.64,car|weekend|female


{'dataset_id': 'flow-dset-links-001',
 'is_validated': False,
 'events': [{'op': 'build_flows',
   'ts_utc': '2026-04-01T12:00:00Z',
   'parameters': {'h3_resolution': 8,
    'group_by': ['mode', 'day_type', 'user_gender'],
    'time_aggregation': 'hour',
    'time_basis': 'origin',
    'min_trips_per_flow': 1},
   'summary': {'n_flows': 240,
    'n_trips_in': 960,
    'n_trips_aggregated': 960,
    'n_trips_dropped': 0,
    'n_flow_to_trips_rows': 720},
   'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
    'top_codes': []}},
  {'op': 'write_flows',
   'ts_utc': '2026-05-20T07:15:25Z',
   'parameters': {'path': 'tmp_integration_read_flows\\artifacts\\case_02_read_with_aux_parquet\\flows_with_aux',
    'mode': 'error_if_exists',
    'storage_format': 'parquet',
    'parquet_compression': 'snappy',
    'feather_compression': 'lz4',
    'normalize_artifact_dir': False,
    'write_flow_to_trips': True},
   'summary': {'n_flows': 240,
    'n_flow_to_trips': 720,
    'pa

{'n_flows': 240,
 'n_columns': 15,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 720,
 'files_read': ['flows.parquet',
  'flow_to_trips.parquet',
  'flows.metadata.json'],
 'dataset_id': 'flow-dset-links-001',
 'artifact_id': 'art_aa724220-5d17-4676-8c53-56f20060e273'}

OK - Bloque 2 - read con auxiliar solicitado y existente


## Bloque 3 - read degradado con auxiliar Parquet solicitado pero faltante

Qué prueba:

- warning/degradación controlada cuando el archivo auxiliar fue solicitado,
  pero ya no está en disco;
- operación retornable bajo `strict=False`;
- issue `READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING`;
- `flow_to_trips=None`;
- summary consistente.

Este comportamiento es coherente con la política de OP-11:
el auxiliar puede faltar de forma recuperable bajo `strict=False`,
pero el sidecar formal no. 

In [10]:
case_dir = make_case_dir("case_03_read_missing_aux_parquet")
artifact_path = case_dir / "flows_missing_aux"

flows = copy.deepcopy(flowdataset_with_trip_links)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert (artifact_path / artifact_aux_filename("parquet")).exists()

# Simulo pérdida del auxiliar
(artifact_path / artifact_aux_filename("parquet")).unlink()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING" in codes
assert loaded.flow_to_trips is None
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None

display(loaded.flows)
display(loaded.metadata)
display(read_report.summary)
display(read_report.issues)
show_ok("Bloque 3 - read degradado con auxiliar Parquet faltante")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,f_00_00235,8828308287fffff,8828308299fffff,19,21.85,metro,leisure,weekend,male,3,midday,2026-04-01 11:00:00+00:00,2026-04-01 12:00:00+00:00,1.01,metro|weekend|male
236,f_00_00236,8828308287fffff,8828308299fffff,20,25.00,car,work,weekday,female,5,afternoon_peak,2026-04-01 12:00:00+00:00,2026-04-01 13:00:00+00:00,1.22,car|weekday|female
237,f_00_00237,8828308287fffff,8828308299fffff,21,26.25,car,education,weekday,male,1,morning_peak,2026-04-01 13:00:00+00:00,2026-04-01 14:00:00+00:00,1.43,car|weekday|male
238,f_00_00238,8828308287fffff,8828308299fffff,5,6.25,car,shopping,weekend,female,3,midday,2026-04-01 14:00:00+00:00,2026-04-01 15:00:00+00:00,1.64,car|weekend|female


{'dataset_id': 'flow-dset-links-001',
 'is_validated': False,
 'events': [{'op': 'build_flows',
   'ts_utc': '2026-04-01T12:00:00Z',
   'parameters': {'h3_resolution': 8,
    'group_by': ['mode', 'day_type', 'user_gender'],
    'time_aggregation': 'hour',
    'time_basis': 'origin',
    'min_trips_per_flow': 1},
   'summary': {'n_flows': 240,
    'n_trips_in': 960,
    'n_trips_aggregated': 960,
    'n_trips_dropped': 0,
    'n_flow_to_trips_rows': 720},
   'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
    'top_codes': []}},
  {'op': 'write_flows',
   'ts_utc': '2026-05-20T07:16:24Z',
   'parameters': {'path': 'tmp_integration_read_flows\\artifacts\\case_03_read_missing_aux_parquet\\flows_missing_aux',
    'mode': 'error_if_exists',
    'storage_format': 'parquet',
    'parquet_compression': 'snappy',
    'feather_compression': 'lz4',
    'normalize_artifact_dir': False,
    'write_flow_to_trips': True},
   'summary': {'n_flows': 240,
    'n_flow_to_trips': 720,
 

{'n_flows': 240,
 'n_columns': 15,
 'flow_to_trips_loaded': False,
 'n_flow_to_trips': None,
 'files_read': ['flows.parquet', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-links-001',
 'artifact_id': 'art_8fb2f1a6-54b8-4c6f-aab4-3628da2d763e'}

[Issue(level='warning', code='READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING', message='Se solicitó cargar flow_to_trips, pero el archivo no existe; la lectura continuará sin auxiliar bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_read_flows\\artifacts\\case_03_read_missing_aux_parquet\\flows_missing_aux', 'read_flow_to_trips': True, 'files_expected': ['flow_to_trips.parquet'], 'files_read': ['flows.parquet', 'flows.metadata.json'], 'reason': 'missing_flow_to_trips_file', 'recovered': True, 'recovery_action': 'omit_missing_flow_to_trips'})]

OK - Bloque 3 - read degradado con auxiliar Parquet faltante


## Bloque 4 - read degradado con sidecar incompleto bajo `strict=False`

Qué prueba:

- recuperación controlada de `dataset_id`;
- degradación de `artifact_id` a `None`;
- degradación de `aggregation_spec` a `{}`;
- continuidad de la lectura bajo `strict=False`;
- emisión de issues explícitos de recuperación.

El sidecar sigue existiendo y es formalmente legible;
solo se corrompen campos recuperables para verificar la matriz de recovery de OP-11.

In [11]:
case_dir = make_case_dir("case_04_read_incomplete_sidecar")
artifact_path = case_dir / "flows_incomplete_sidecar"

flows = copy.deepcopy(flowdataset_small)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

assert write_report.ok is True

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

original_dataset_id = sidecar["dataset_id"]
original_artifact_id = sidecar["artifact_id"]

# Corrompo solo partes recuperables bajo strict=False
sidecar["dataset_id"] = ""
sidecar["artifact_id"] = None
sidecar["aggregation_spec"] = None

sidecar_path.write_text(
    json.dumps(sidecar, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.METADATA.DATASET_ID_REGENERATED" in codes
assert "READ_FLOWS.METADATA.ARTIFACT_ID_SET_NONE" in codes
assert "READ_FLOWS.SIDECAR.AGGREGATION_SPEC_DEFAULTED" in codes

assert loaded.metadata["dataset_id"] != original_dataset_id
assert loaded.metadata["dataset_id"] is not None
assert loaded.metadata["artifact_id"] is None
assert loaded.aggregation_spec == {}

display(read_report.issues)
show_ok("Bloque 4 - read degradado con sidecar incompleto y strict=False")

[Issue(level='warning', code='READ_FLOWS.METADATA.DATASET_ID_REGENERATED', message='dataset_id faltaba o era inválido en el sidecar; se regeneró bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_read_flows\\artifacts\\case_04_read_incomplete_sidecar\\flows_incomplete_sidecar', 'reason': 'invalid_dataset_id', 'recovered': True, 'recovery_action': 'regenerate_dataset_id', 'dataset_id_status': 'regenerated'}),
 Issue(level='warning', code='READ_FLOWS.METADATA.ARTIFACT_ID_SET_NONE', message='artifact_id faltaba o era inválido en el sidecar; se degradó a None bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_read_flows\\artifacts\\case_04_read_incomplete_sidecar\\flows_incomplete_sidecar', 'reason': 'invalid_artifact_id', 'recovered': True, 'recovery_action': 'set_artifact_id_none', 'artifact_id_status': 'set_none'}),
 Issue(level='warning', code='READ_FLOWS.SIDECAR.AGGREGATION_SPEC_DEFAU

OK - Bloque 4 - read degradado con sidecar incompleto y strict=False


## Bloque 5 - layout fatal sin sidecar

Qué prueba:

- la lectura formal debe fallar si falta `flows.metadata.json`;
- una tabla física `flows.parquet` por sí sola no constituye
  un artefacto legible por `read_flows`;
- se lanza `ExportError`.

El sidecar es obligatorio para la persistencia formal de flows. 

In [12]:
case_dir = make_case_dir("case_05_layout_fatal_missing_sidecar")
artifact_path = case_dir / "flows_without_sidecar"
artifact_path.mkdir(parents=True, exist_ok=True)

flowdataset_small.flows.to_parquet(
    artifact_path / "flows.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None

try:
    read_flows(
        artifact_path,
        options=ReadFlowsOptions(
            strict=False,
            keep_metadata=True,
            read_flow_to_trips=False,
        ),
    )
except Exception as exc:
    raised = exc

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Bloque 5 - layout fatal sin sidecar")

ExportError(message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', details={'path': 'tmp_integration_read_flows\\artifacts\\case_05_layout_fatal_missing_sidecar\\flows_without_sidecar', 'files_expected': ['flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}, issue=Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_read_flows\\artifacts\\case_05_layout_fatal_missing_sidecar\\flows_without_sidecar', 'files_expected': ['flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}), issues=(Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es rec

OK - Bloque 5 - layout fatal sin sidecar


## Bloque 6 - round-trip Parquet rico + política `is_validated=False` post-read

Qué prueba:

- write/read encadenados con fixture rica;
- comparación estructural de:
  - `flows`,
  - `flow_to_trips`,
  - `aggregation_spec`,
  - `provenance`;
- preservación de `dataset_id`;
- recuperación de `artifact_id`;
- `source_trips=None` post-read;
- issue `READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE`
  cuando el dataset persistido venía con `is_validated=True`.

La política de OP-11 es que toda lectura formal fuerza
`metadata["is_validated"] = False`, porque leer no equivale a revalidar. 

In [13]:
case_dir = make_case_dir("case_06_roundtrip_parquet_policy")
artifact_path = case_dir / "flows_roundtrip"

flows = make_rich_flowdataset(
    repeat_blocks=2,
    with_trip_links=True,
    validated=True,   # intencional para probar la política post-read
    dataset_id="flow-dset-roundtrip-parquet-001",
)

flows_before = flows.flows.copy(deep=True)
flow_to_trips_before = flows.flow_to_trips.copy(deep=True)
aggregation_before = copy.deepcopy(flows.aggregation_spec)
provenance_before = copy.deepcopy(flows.provenance)
dataset_id_before = flows.metadata["dataset_id"]

display(flows.flows)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE" in codes

assert_df_equal_untyped(
    loaded.flows,
    flows_before,
    by=["flow_id"],
)

assert_df_equal_untyped(
    loaded.flow_to_trips,
    flow_to_trips_before,
    by=["flow_id", "movement_id"],
)

assert loaded.aggregation_spec == aggregation_before
assert loaded.provenance == provenance_before
assert loaded.metadata["dataset_id"] == dataset_id_before
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

display(loaded.flows)

print("write summary:")
display(write_report.summary)

print("read summary:")
display(read_report.summary)

show_ok("Bloque 6 - round-trip Parquet rico + política is_validated=False post-read")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,f_01_00475,8828308287fffff,8828308299fffff,21,24.15,metro,leisure,weekend,male,3,midday,2026-04-02 11:00:00+00:00,2026-04-02 12:00:00+00:00,2.27,metro|weekend|male
476,f_01_00476,8828308287fffff,8828308299fffff,5,6.25,car,work,weekday,female,5,afternoon_peak,2026-04-02 12:00:00+00:00,2026-04-02 13:00:00+00:00,2.48,car|weekday|female
477,f_01_00477,8828308287fffff,8828308299fffff,6,7.50,car,education,weekday,male,1,morning_peak,2026-04-02 13:00:00+00:00,2026-04-02 14:00:00+00:00,0.80,car|weekday|male
478,f_01_00478,8828308287fffff,8828308299fffff,7,8.75,car,shopping,weekend,female,3,midday,2026-04-02 14:00:00+00:00,2026-04-02 15:00:00+00:00,1.01,car|weekend|female


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,f_01_00475,8828308287fffff,8828308299fffff,21,24.15,metro,leisure,weekend,male,3,midday,2026-04-02 11:00:00+00:00,2026-04-02 12:00:00+00:00,2.27,metro|weekend|male
476,f_01_00476,8828308287fffff,8828308299fffff,5,6.25,car,work,weekday,female,5,afternoon_peak,2026-04-02 12:00:00+00:00,2026-04-02 13:00:00+00:00,2.48,car|weekday|female
477,f_01_00477,8828308287fffff,8828308299fffff,6,7.50,car,education,weekday,male,1,morning_peak,2026-04-02 13:00:00+00:00,2026-04-02 14:00:00+00:00,0.80,car|weekday|male
478,f_01_00478,8828308287fffff,8828308299fffff,7,8.75,car,shopping,weekend,female,3,midday,2026-04-02 14:00:00+00:00,2026-04-02 15:00:00+00:00,1.01,car|weekend|female


write summary:


{'n_flows': 480,
 'n_flow_to_trips': 1440,
 'files_written': ['flows.parquet',
  'flows.metadata.json',
  'flow_to_trips.parquet'],
 'dataset_id': 'flow-dset-roundtrip-parquet-001',
 'artifact_id': 'art_324c29d2-2250-4ea3-bd10-b76b14b16320',
 'path': 'tmp_integration_read_flows\\artifacts\\case_06_roundtrip_parquet_policy\\flows_roundtrip'}

read summary:


{'n_flows': 480,
 'n_columns': 15,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 1440,
 'files_read': ['flows.parquet',
  'flow_to_trips.parquet',
  'flows.metadata.json'],
 'dataset_id': 'flow-dset-roundtrip-parquet-001',
 'artifact_id': 'art_324c29d2-2250-4ea3-bd10-b76b14b16320'}

OK - Bloque 6 - round-trip Parquet rico + política is_validated=False post-read


## Bloque 7 - read feliz desde bundle Feather

Qué prueba:

- reconstrucción correcta desde un bundle Feather formal;
- fallback a `.golondrina`;
- `files_read` coherente con `flows.feather`;
- reconstrucción de `flows`, `aggregation_spec`, `provenance` y metadata;
- evento `read_flows`;
- política `is_validated=False` post-read.

Este caso es importante porque Feather quedó como backend por defecto de escritura,
pero `read_flows` no recibe el backend por argumento: lo resuelve desde el sidecar. 

In [14]:
case_dir = make_case_dir("case_07_read_happy_feather")
artifact_path = case_dir / "flows_read_happy_feather"

flows = copy.deepcopy(flowdataset_small)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,  # sin sufijo; debe usar fallback a .golondrina
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

effective_root = Path(str(artifact_path) + ".golondrina")

assert read_report.ok is True

# Parameters
assert read_report.parameters["path"] == str(effective_root)
assert read_report.parameters["strict"] is False
assert read_report.parameters["keep_metadata"] is True
assert read_report.parameters["read_flow_to_trips"] is False

# Summary
assert read_report.summary["n_flows"] == len(flows.flows)
assert read_report.summary["n_columns"] == len(flows.flows.columns)
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert set(read_report.summary["files_read"]) == {
    "flows.feather",
    "flows.metadata.json",
}

# Dataset reconstruido
assert_df_equal_untyped(
    loaded.flows,
    flows.flows,
    by=["flow_id"],
)

assert loaded.aggregation_spec == flows.aggregation_spec
assert loaded.provenance == flows.provenance
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

# Evento read
event = loaded.metadata["events"][-1]
assert event["op"] == "read_flows"
assert event["parameters"] == read_report.parameters
assert event["summary"] == read_report.summary
assert "issues_summary" in event

display(read_report)
show_ok("Bloque 7 - read feliz desde bundle Feather")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_columns': 15, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_1818dbaf-fdc9-4c62-9790-b4c713a87ea1'}, parameters={'path': 'tmp_integration_read_flows\\artifacts\\case_07_read_happy_feather\\flows_read_happy_feather.golondrina', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': False})

OK - Bloque 7 - read feliz desde bundle Feather


## Bloque 8 - round-trip Feather con auxiliar presente

Qué prueba:

- write/read completo con backend Feather;
- persistencia de:
  - `flows.feather`,
  - `flow_to_trips.feather`,
  - `flows.metadata.json`;
- lectura del auxiliar;
- igualdad estructural de las tablas reconstruidas;
- política `metadata["is_validated"] = False` post-read;
- summary de lectura con `files_read` coherente.

Este bloque es la contraparte Feather del round-trip rico de Parquet.

In [15]:
case_dir = make_case_dir("case_08_roundtrip_feather_with_aux")
artifact_path = case_dir / "flows_roundtrip_feather"

flows = make_rich_flowdataset(
    repeat_blocks=2,
    with_trip_links=True,
    validated=True,
    dataset_id="flow-dset-roundtrip-feather-001",
)

flows_before = flows.flows.copy(deep=True)
flow_to_trips_before = flows.flow_to_trips.copy(deep=True)
aggregation_before = copy.deepcopy(flows.aggregation_spec)
provenance_before = copy.deepcopy(flows.provenance)
dataset_id_before = flows.metadata["dataset_id"]

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert (artifact_path / "flows.feather").exists()
assert (artifact_path / "flow_to_trips.feather").exists()
assert set(write_report.summary["files_written"]) == {
    "flows.feather",
    "flow_to_trips.feather",
    "flows.metadata.json",
}

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE" in codes

assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)
assert set(read_report.summary["files_read"]) == {
    "flows.feather",
    "flow_to_trips.feather",
    "flows.metadata.json",
}

assert_df_equal_untyped(
    loaded.flows,
    flows_before,
    by=["flow_id"],
)

assert_df_equal_untyped(
    loaded.flow_to_trips,
    flow_to_trips_before,
    by=["flow_id", "movement_id"],
)

assert loaded.aggregation_spec == aggregation_before
assert loaded.provenance == provenance_before
assert loaded.metadata["dataset_id"] == dataset_id_before
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

display(write_report.summary)
display(read_report.summary)
show_ok("Bloque 8 - round-trip Feather con auxiliar presente")

{'n_flows': 480,
 'n_flow_to_trips': 1440,
 'files_written': ['flows.feather',
  'flows.metadata.json',
  'flow_to_trips.feather'],
 'dataset_id': 'flow-dset-roundtrip-feather-001',
 'artifact_id': 'art_11340397-a03b-4983-a925-36a64f163b56',
 'path': 'tmp_integration_read_flows\\artifacts\\case_08_roundtrip_feather_with_aux\\flows_roundtrip_feather'}

{'n_flows': 480,
 'n_columns': 15,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 1440,
 'files_read': ['flows.feather',
  'flow_to_trips.feather',
  'flows.metadata.json'],
 'dataset_id': 'flow-dset-roundtrip-feather-001',
 'artifact_id': 'art_11340397-a03b-4983-a925-36a64f163b56'}

OK - Bloque 8 - round-trip Feather con auxiliar presente


## Bloque 9 - read degradado con auxiliar Feather solicitado pero faltante

Qué prueba:

- warning/degradación controlada cuando `flow_to_trips.feather`
  fue pedido pero no está en disco;
- operación retornable bajo `strict=False`;
- issue `READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING`;
- `flow_to_trips=None`;
- `files_read` solo contiene tabla principal y sidecar.

In [16]:
case_dir = make_case_dir("case_09_read_missing_aux_feather")
artifact_path = case_dir / "flows_missing_aux_feather"

flows = copy.deepcopy(flowdataset_with_trip_links)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert (artifact_path / "flow_to_trips.feather").exists()

# Simulo pérdida del auxiliar Feather
(artifact_path / "flow_to_trips.feather").unlink()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING" in codes
assert loaded.flow_to_trips is None
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert set(read_report.summary["files_read"]) == {
    "flows.feather",
    "flows.metadata.json",
}

display(read_report.issues)
display(read_report.summary)
show_ok("Bloque 9 - read degradado con auxiliar Feather solicitado pero faltante")

[Issue(level='warning', code='READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING', message='Se solicitó cargar flow_to_trips, pero el archivo no existe; la lectura continuará sin auxiliar bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_read_flows\\artifacts\\case_09_read_missing_aux_feather\\flows_missing_aux_feather', 'read_flow_to_trips': True, 'files_expected': ['flow_to_trips.feather'], 'files_read': ['flows.feather', 'flows.metadata.json'], 'reason': 'missing_flow_to_trips_file', 'recovered': True, 'recovery_action': 'omit_missing_flow_to_trips'})]

{'n_flows': 240,
 'n_columns': 15,
 'flow_to_trips_loaded': False,
 'n_flow_to_trips': None,
 'files_read': ['flows.feather', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-links-001',
 'artifact_id': 'art_9c445a18-379a-4860-935c-998edf02f01b'}

OK - Bloque 9 - read degradado con auxiliar Feather solicitado pero faltante


## Bloque 10 - mismatch fatal entre `storage.format="feather"` y `files.data`

Qué prueba:

- inconsistencia no recuperable entre backend declarado y nombre físico del archivo principal;
- el sidecar declara `storage.format="feather"`;
- luego se fuerza `files.data="flows.parquet"`;
- con `strict=True`, `read_flows` debe abortar;
- el error expuesto corresponde a `READ_FLOWS.LAYOUT.MISSING_DATA_FILE`.

Esto valida que OP-11 no solo confía ciegamente en el sidecar,
sino que también comprueba coherencia entre backend y archivos físicos declarados. 

In [17]:
case_dir = make_case_dir("case_10_sidecar_mismatch_feather")
artifact_path = case_dir / "flows_sidecar_mismatch_feather"

flows = copy.deepcopy(flowdataset_small)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

assert write_report.ok is True
assert (artifact_path / "flows.feather").exists()

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

# Fuerzo inconsistencia:
# sidecar declara backend feather, pero apunta al nombre físico de parquet.
sidecar["files"]["data"] = "flows.parquet"

sidecar_path.write_text(
    json.dumps(sidecar, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

raised = None

try:
    read_flows(
        artifact_path,
        options=ReadFlowsOptions(
            strict=True,
            keep_metadata=True,
            read_flow_to_trips=False,
        ),
    )
except Exception as exc:
    raised = exc

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ_FLOWS.LAYOUT.MISSING_DATA_FILE"

display(raised)
show_ok("Bloque 10 - mismatch fatal entre storage.format y files.data")

ExportError(message='El bundle de flows no contiene o no resuelve correctamente el archivo principal de datos requerido.', code='READ_FLOWS.LAYOUT.MISSING_DATA_FILE', details={'path': 'tmp_integration_read_flows\\artifacts\\case_10_sidecar_mismatch_feather\\flows_sidecar_mismatch_feather', 'files_expected': ['flows.feather'], 'reason': "data_file_mismatch: declared='flows.parquet' expected='flows.feather'", 'action': 'abort'}, issue=Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_DATA_FILE', message='El bundle de flows no contiene o no resuelve correctamente el archivo principal de datos requerido.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_read_flows\\artifacts\\case_10_sidecar_mismatch_feather\\flows_sidecar_mismatch_feather', 'files_expected': ['flows.feather'], 'reason': "data_file_mismatch: declared='flows.parquet' expected='flows.feather'", 'action': 'abort'}), issues=(Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_DATA_FILE', me

OK - Bloque 10 - mismatch fatal entre storage.format y files.data


## Bloque 11 - read de bundle Parquet formal sigue funcionando

Qué prueba:

- compatibilidad de lectura Parquet;
- aunque Feather sea el backend por defecto de escritura,
  `read_flows` debe seguir reconstruyendo correctamente
  un artefacto formal Parquet guiándose por el sidecar;
- lectura de `flow_to_trips.parquet`;
- summary, dataset, metadata y evento coherentes.

In [18]:
case_dir = make_case_dir("case_11_read_formal_parquet_still_works")
artifact_path = case_dir / "flows_read_formal_parquet"

flows = copy.deepcopy(flowdataset_with_trip_links)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert (artifact_path / "flows.parquet").exists()
assert (artifact_path / "flow_to_trips.parquet").exists()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert read_report.ok is True
assert set(read_report.summary["files_read"]) == {
    "flows.parquet",
    "flow_to_trips.parquet",
    "flows.metadata.json",
}
assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)

assert_df_equal_untyped(
    loaded.flows,
    flows.flows,
    by=["flow_id"],
)

assert_df_equal_untyped(
    loaded.flow_to_trips,
    flows.flow_to_trips,
    by=["flow_id", "movement_id"],
)

assert loaded.aggregation_spec == flows.aggregation_spec
assert loaded.provenance == flows.provenance
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

event = loaded.metadata["events"][-1]
assert event["op"] == "read_flows"
assert event["parameters"] == read_report.parameters
assert event["summary"] == read_report.summary
assert "issues_summary" in event

display(read_report)
show_ok("Bloque 11 - read de bundle Parquet formal sigue funcionando")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_columns': 15, 'flow_to_trips_loaded': True, 'n_flow_to_trips': 720, 'files_read': ['flows.parquet', 'flow_to_trips.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-links-001', 'artifact_id': 'art_708c065c-f13c-44a3-9002-1754b30dfdc4'}, parameters={'path': 'tmp_integration_read_flows\\artifacts\\case_11_read_formal_parquet_still_works\\flows_read_formal_parquet', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': True})

OK - Bloque 11 - read de bundle Parquet formal sigue funcionando
